In [5]:
from pathlib import Path
import sys

import openpyxl
import pandas as pd
from openpyxl.styles import PatternFill

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def project_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _build_article_lookup(df1: pd.DataFrame) -> tuple[dict[str, int], dict[str, tuple[str, ...]]]:
    article_to_group: dict[str, int] = {}
    article_to_analogs_sets: dict[str, set[str]] = {}

    for _, row in df1.iterrows():
        group = row["Номер группы"]
        analogs = row["all_analogs"]
        if pd.isna(group) or not isinstance(analogs, tuple):
            continue

        forms_to_bind = set(analogs)
        main = row.get("Номенклатура.Артикул")
        if pd.notna(main) and main:
            forms_to_bind.add(main)

        for art in forms_to_bind:
            article_to_group[art] = int(group)
            article_to_analogs_sets.setdefault(art, set()).update(analogs)

    article_to_analogs = {
        art: tuple(sorted(analogs_set))
        for art, analogs_set in article_to_analogs_sets.items()
    }

    return article_to_group, article_to_analogs


GROUP_FILLS = [
    "FDE68A",
    "BFDBFE",
    "C7D2FE",
    "BBF7D0",
    "FBCFE8",
    "A7F3D0",
    "FECACA",
    "DDD6FE",
    "FED7AA",
    "E9D5FF",
]


from pipeline.runner import (
    _build_article_lookup,
    _drop_rows_without_identifiers,
    _enrich_analogs,
    _lookup_group,
    _sync_group_membership,
)
from preprocessing.cleaning import (
    fill_missing_article,
    normalize_nomenclatures_repair_parts,
    normalize_nomenclatures_stock_report,
)
from preprocessing.grouping import (
    build_analog_graph,
    consolidate_extended_article_numbers,
    find_all_analogs,
    normalize_analog_lists,
)
from preprocessing.normalization import article_forms, normalize
from readers.loaders import preprocess_repair_parts, preprocess_stock_report


def build_group_analogs_excel(
    repair_path: str | Path,
    stock_path: str | Path,
    output_path: str | Path = "data/processed/Аналоги.xlsx",
) -> pd.DataFrame:
    repair_path = project_path(repair_path)
    stock_path = project_path(stock_path)
    output_path = project_path(output_path)

    # Логика построения групп полностью повторяет pipeline.runner.run_full_pipeline.
    df1 = preprocess_repair_parts(str(repair_path))
    df1 = normalize_nomenclatures_repair_parts(df1)

    for col in [
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
        "Номенклатура.Оригинальный номер расширенный",
    ]:
        df1[col] = df1[col].apply(normalize)

    df1 = fill_missing_article(
        df1,
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
    )
    df1 = _drop_rows_without_identifiers(
        df1,
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
    )

    df1["Номенклатура.Оригинальный номер расширенный"] = df1.apply(
        consolidate_extended_article_numbers,
        axis=1,
    )
    df1["Аналоги"] = (
        df1["Номенклатура.Оригинальный номер расширенный"]
        .fillna("")
        .str.upper()
        .str.split()
    )

    graph = build_analog_graph(df1)
    df1["all_analogs"] = df1["Номенклатура.Артикул"].apply(
        lambda x: find_all_analogs(x, graph)
    )
    df1 = df1.drop(columns="Аналоги")

    group_mapping = {
        analog_tuple: idx
        for idx, analog_tuple in enumerate(df1["all_analogs"].unique(), start=1)
    }
    df1["Номер группы"] = df1["all_analogs"].apply(group_mapping.get).astype("Int64")

    df2 = preprocess_stock_report(str(stock_path))
    df2 = normalize_nomenclatures_stock_report(df2)
    df2["Артикул"] = df2["Артикул"].apply(normalize)
    df2["Оригинальный номер"] = df2["Оригинальный номер"].apply(normalize)
    df2 = fill_missing_article(df2, "Артикул", "Оригинальный номер")
    df2 = _drop_rows_without_identifiers(df2, "Артикул", "Оригинальный номер")

    article_to_group, article_to_analogs = _build_article_lookup(df1)

    groups, analogs_list = zip(
        *df2.apply(
            lambda r: _lookup_group(
                r["Артикул"],
                r["Оригинальный номер"],
                article_to_group,
                article_to_analogs,
            ),
            axis=1,
        )
    )
    df2["Номер группы"] = pd.Series(groups, index=df2.index, dtype="Int64")
    df2["Список аналогов"] = analogs_list

    df2["Список аналогов"] = df2.apply(
        lambda r: _enrich_analogs(r, article_to_group),
        axis=1,
    )
    df2 = normalize_analog_lists(df2)

    df1, df2 = _sync_group_membership(df1, df2)
    article_to_group, article_to_analogs = _build_article_lookup(df1)

    unmatched_idx = df2[df2["Номер группы"].isna()].index
    relinked_idx = []

    for idx in unmatched_idx:
        grp, analogs = _lookup_group(
            normalize(df2.at[idx, "Артикул"]),
            normalize(df2.at[idx, "Оригинальный номер"]),
            article_to_group,
            article_to_analogs,
        )
        if grp is None:
            continue

        df2.at[idx, "Номер группы"] = grp
        df2.at[idx, "Список аналогов"] = analogs
        relinked_idx.append(idx)

    if relinked_idx:
        df2.loc[relinked_idx, "Список аналогов"] = (
            df2.loc[relinked_idx]
            .apply(lambda r: _enrich_analogs(r, article_to_group), axis=1)
        )
        df2 = normalize_analog_lists(df2)
        df1, df2 = _sync_group_membership(df1, df2)
        article_to_group, article_to_analogs = _build_article_lookup(df1)

    graph_new = {}
    graph_new = __import__("collections").defaultdict(set)
    unmatched_idx = df2[df2["Номер группы"].isna()].index

    for idx in unmatched_idx:
        art = normalize(df2.at[idx, "Артикул"])
        orig = normalize(df2.at[idx, "Оригинальный номер"])

        if art is None:
            if orig is not None:
                art = orig
            else:
                continue

        all_forms = article_forms(art) + (article_forms(orig) if orig else [])
        if any(f in article_to_group for f in all_forms):
            continue

        for af in article_forms(art):
            for bf in article_forms(art):
                if af != bf:
                    graph_new[af].add(bf)
                    graph_new[bf].add(af)

        if orig is not None:
            for af in article_forms(art):
                for bf in article_forms(orig):
                    if af != bf:
                        graph_new[af].add(bf)
                        graph_new[bf].add(af)

    for idx in unmatched_idx:
        art = normalize(df2.at[idx, "Артикул"])
        orig = normalize(df2.at[idx, "Оригинальный номер"])
        if art is None:
            art = orig

        all_forms = article_forms(art) if art else []
        if any(f in article_to_group for f in all_forms):
            continue

        df2.at[idx, "Список аналогов"] = (
            find_all_analogs(art, graph_new) if art is not None else tuple()
        )

    unique_new = (
        df2.loc[df2["Номер группы"].isna(), "Список аналогов"]
        .drop_duplicates()
    )
    new_group_start = int(df1["Номер группы"].max()) + 1
    new_group_map = {
        grp: new_group_start + i
        for i, grp in enumerate(unique_new)
    }
    mask_new = df2["Номер группы"].isna()
    df2.loc[mask_new, "Номер группы"] = df2.loc[mask_new, "Список аналогов"].apply(
        lambda x: new_group_map.get(x)
    )
    df2["Номер группы"] = df2["Номер группы"].astype("Int64")
    df2 = normalize_analog_lists(df2)

    repair_members = df1[[
        "Номер группы",
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
        "Номенклатура.Оригинальный номер расширенный",
        "Номенклатура",
        "all_analogs",
    ]].rename(columns={
        "Номенклатура.Артикул": "Артикул",
        "Номенклатура.Оригинальный номер": "Оригинальный номер",
        "Номенклатура.Оригинальный номер расширенный": "Оригинальный номер расширенный",
        "all_analogs": "Список аналогов",
    })

    stock_members = df2[[
        "Номер группы",
        "Артикул",
        "Оригинальный номер",
        "Номенклатура",
        "Список аналогов",
    ]]
    stock_members["Оригинальный номер расширенный"] = pd.NA

    members = pd.concat([repair_members, stock_members], ignore_index=True)
    members = members[members["Номер группы"].notna()].copy()
    members["Номер группы"] = members["Номер группы"].astype(int)
    members = normalize_analog_lists(members)

    members["Артикул_key"] = (
        members["Артикул"].astype("string").str.strip().replace("", pd.NA)
    )
    members["Оригинальный номер_key"] = (
        members["Оригинальный номер"].astype("string").str.strip().replace("", pd.NA)
    )
    members["Оригинальный номер расширенный_key"] = (
        members["Оригинальный номер расширенный"].astype("string").str.strip().replace("", pd.NA)
    )
    members["Номенклатура_key"] = (
        members["Номенклатура"].astype("string").str.strip().replace("", pd.NA)
    )

    group_stats = (
        members.groupby("Номер группы", as_index=False)
        .agg(
            unique_articles=("Артикул_key", lambda s: s.dropna().nunique()),
            unique_original_numbers=("Оригинальный номер_key", lambda s: s.dropna().nunique()),
            unique_extended_original_numbers=("Оригинальный номер расширенный_key", lambda s: s.dropna().nunique()),
            unique_nomenclatures=("Номенклатура_key", lambda s: s.dropna().nunique()),
        )
    )

    valid_groups = group_stats.loc[
        (group_stats["unique_articles"] > 1)
        | (group_stats["unique_original_numbers"] > 1)
        | (group_stats["unique_extended_original_numbers"] > 1)
        | (group_stats["unique_nomenclatures"] > 1),
        "Номер группы",
    ]
    members = members[members["Номер группы"].isin(valid_groups)].copy()

    distinct_members = members.drop_duplicates(
        subset=[
            "Номер группы",
            "Артикул_key",
            "Оригинальный номер_key",
            "Оригинальный номер расширенный_key",
            "Номенклатура_key",
        ],
        keep="first",
    ).copy()

    full_analogs_map = (
        members.groupby("Номер группы")["Список аналогов"]
        .agg(
            lambda s: max(
                (v for v in s.dropna() if isinstance(v, tuple)),
                key=len,
                default=tuple(),
            )
        )
        .to_dict()
    )

    export_df = distinct_members[[
        "Номер группы",
        "Артикул",
        "Оригинальный номер",
        "Оригинальный номер расширенный",
        "Номенклатура",
    ]].copy()
    export_df["Список аналогов"] = export_df["Номер группы"].map(
        lambda group_id: " ".join(full_analogs_map.get(group_id, tuple()))
    )
    export_df = export_df.sort_values(
        [
            "Номер группы",
            "Артикул",
            "Оригинальный номер",
            "Оригинальный номер расширенный",
            "Номенклатура",
        ],
        kind="stable",
    ).reset_index(drop=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    export_df.to_excel(output_path, index=False)

    workbook = openpyxl.load_workbook(output_path)
    worksheet = workbook.active
    group_colors = {
        group_id: GROUP_FILLS[idx % len(GROUP_FILLS)]
        for idx, group_id in enumerate(export_df["Номер группы"].drop_duplicates())
    }

    for row_idx, group_id in enumerate(export_df["Номер группы"], start=2):
        fill = PatternFill(fill_type="solid", fgColor=group_colors[group_id])
        for col_idx in range(1, worksheet.max_column + 1):
            worksheet.cell(row=row_idx, column=col_idx).fill = fill

    workbook.save(output_path)

    return export_df


export_df = build_group_analogs_excel(
    repair_path = r"C:\Users\a.vorona\Desktop\Forecast\data\raw\Запчасти списанные в ремонт_30_03_26.xlsx",
    stock_path = r"C:\Users\a.vorona\Desktop\Forecast\data\raw\Остатки и обороты_30_03_26.xlsx",
    output_path = "data/processed/Аналоги.xlsx",
)

export_df.head(20)

C:\Users\a.vorona\AppData\Local\Temp\ipykernel_11456\3612583230.py:277: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock_members["Оригинальный номер расширенный"] = pd.NA


,Номер группы,Артикул,Оригинальный номер,Оригинальный номер расширенный,Номенклатура,Список аналогов
0,9,1020521389,1020521389,26.01.000023,Замок зажигания с ключом IHP-30 (1020520943) 1...,1020521389 26.01.000023 26.01.000024 413000072...
1,9,1020521389,1020521389,None,Замок зажигания с ключом IHP-30 (1020520943) 1...,1020521389 26.01.000023 26.01.000024 413000072...
2,9,4130000723,4130000723,4130000723001,Замок зажигания 4130000723,1020521389 26.01.000023 26.01.000024 413000072...
3,9,4130000723,4130000723,NaN,Замок зажигания 4130000723,1020521389 26.01.000023 26.01.000024 413000072...
4,9,4130000723001,4130000723001,26.01.000023 89943-29 Y26.01.000023,Замок зажигания с ключом 84830/90173 41300007...,1020521389 26.01.000023 26.01.000024 413000072...
5,9,4130000723001,4130000723001,None,Замок зажигания 4130000723001 LGMG,1020521389 26.01.000023 26.01.000024 413000072...
6,9,4130000723001,4130000723001,None,Замок зажигания 84830/90173 4130000723001 LGMG,1020521389 26.01.000023 26.01.000024 413000072...
7,9,4130000723001,4130000723001,NaN,Замок зажигания с ключом 84830/90173 41300007...,1020521389 26.01.000023 26.01.000024 413000072...
8,9,84830,26.01.000024,1020521389 26.01.000024 IHP-30 Y26.01.000024,Замок зажигания с ключом 84830 ZOOMLION,1020521389 26.01.000023 26.01.000024 413000072...
9,9,84830,26.01.000024,NaN,Замок зажигания с ключом 84830 ZOOMLION,1020521389 26.01.000023 26.01.000024 413000072...


In [7]:
df1 = preprocess_repair_parts(r"C:\Users\a.vorona\Desktop\Forecast\data\raw\Запчасти списанные в ремонт_30_03_26.xlsx")
df1 = normalize_nomenclatures_repair_parts(df1)
df1.to_excel('text.xlsx', index=False)

df2 = preprocess_stock_report(r"C:\Users\a.vorona\Desktop\Forecast\data\raw\Остатки и обороты_30_03_26.xlsx")
df2 = normalize_nomenclatures_stock_report(df2)
# df2.to_excel('text.xlsx', index=False)

In [10]:
name = "Фильтр воздушный внешний DIFA 43123 + внутренний DIFA 43123-01"
display(df1.loc[df1['Номенклатура'] == name])
display(df2.loc[df2['Номенклатура'] == name])

,Дата,Год,Месяц,Номенклатура,Номенклатура.Артикул,Номенклатура.Оригинальный номер,Номенклатура.Оригинальный номер расширенный,Машина,Машина_подъемник,Количество


,Год,Месяц,Номенклатура,Артикул,Оригинальный номер,Приход,Расход,Код,Конечный остаток
31377,2025,11,Фильтр воздушный внешний DIFA 43123 + внутренн...,43123,43123,8.0,0.0,Я0408441,8.0
31378,2025,11,Фильтр воздушный внешний DIFA 43123 + внутренн...,43123-01,43123-01,8.0,0.0,Я0408441,8.0
31723,2026,3,Фильтр воздушный внешний DIFA 43123 + внутренн...,43123,43123,0.0,0.0,Я0408441,8.0
31724,2026,3,Фильтр воздушный внешний DIFA 43123 + внутренн...,43123-01,43123-01,0.0,0.0,Я0408441,8.0


In [4]:
s = "12990755801 1399760 2992241 33654 33956 3654 4894548 4897833 533654 65125035026 6754716130 6754716140 6754796130 6754796140 86654 9414101755 A408064 BF7813 BF7957 BF7966 FF165 FF5420 FF5421 FF5485 FF550881 FK28516 FSM4166 H18WK05 LFF5485 P550881 SK3297 SN40529 WK95021" 

lst = s.split(' ')
counts = {}

for word in lst:
    if word in counts:
        counts[word] += 1
    else:
        counts[word] = 1

for word, count in counts.items():
    if count > 1:
        print(word, count)